# SLIMsequence — NER + XLNet-base Fake News Classification


**How to run:** Needs GPU runtime to execute. Public datasets are downloaded automatically; checkpoints and results are written under `/content/SLIM_NER_results/`. Running all will train both models again and take substantial time.

The original NER preprocessing used `set()` to deduplicate entities, so entity order and resulting scores may differ across fresh Python sessions.


In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU memory: 94.97 GB


In [2]:
!pip install -q transformers accelerate sentencepiece nltk

In [3]:
import transformers
import nltk
import numpy as np
import pandas as pd
import sklearn

print("Transformers:", transformers.__version__)
print("NLTK:", nltk.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

Transformers: 5.22.1
NLTK: 3.9.1
NumPy: 2.4.6
Pandas: 2.2.3
Scikit-learn: 1.6.1


In [4]:
# Base URL for the datasets in the original SLIM repository
base_url = "https://raw.githubusercontent.com/kappakant/SLIM/main/dataset"

# Define the original dataset splits
files = {
    "ReCOVery": {
        "train": f"{base_url}/ReCOVery/train_preprocessed.csv",
        "validation": f"{base_url}/ReCOVery/eval_preprocessed.csv",
        "test": f"{base_url}/ReCOVery/test_preprocessed.csv"
    },
    "Fake_And_Real_News": {
        "train": f"{base_url}/fake_and_real_news/train_preprocessed.csv",
        "validation": f"{base_url}/fake_and_real_news/eval_preprocessed.csv",
        "test": f"{base_url}/fake_and_real_news/test_preprocessed.csv"
    }
}

# Load each dataset while preserving the original text and labels
ner_raw = {}

for dataset_name, splits in files.items():
    ner_raw[dataset_name] = {}

    for split_name, url in splits.items():
        df = pd.read_csv(url)

        # Standardize the text column across both datasets
        text_column = (
            "text" if dataset_name == "ReCOVery"
            else "merged_info"
        )

        df = df.rename(columns={text_column: "body"})

        # Retain only the article text and classification label
        ner_raw[dataset_name][split_name] = df[
            ["body", "label"]
        ].copy()

        # Display dataset dimensions
        print(dataset_name, split_name, df.shape[0])

ReCOVery train 1453
ReCOVery validation 184
ReCOVery test 392
Fake_And_Real_News train 2297
Fake_And_Real_News validation 1148
Fake_And_Real_News test 1149


In [5]:
# Inspect class distributions and check for missing values
for dataset_name, splits in ner_raw.items():
    print(f"\nDataset: {dataset_name}")

    for split_name, df in splits.items():
        print(f"\n{split_name.upper()}")

        # Display the number of samples belonging to each class
        print(df["label"].value_counts().sort_index())

        # Check for missing article text and labels
        print("Missing text:", df["body"].isna().sum())
        print("Missing labels:", df["label"].isna().sum())


Dataset: ReCOVery

TRAIN
label
0    487
1    966
Name: count, dtype: int64
Missing text: 0
Missing labels: 0

VALIDATION
label
0     64
1    120
Name: count, dtype: int64
Missing text: 0
Missing labels: 0

TEST
label
0    114
1    278
Name: count, dtype: int64
Missing text: 0
Missing labels: 0

Dataset: Fake_And_Real_News

TRAIN
label
0    1154
1    1143
Name: count, dtype: int64
Missing text: 0
Missing labels: 0

VALIDATION
label
0    551
1    597
Name: count, dtype: int64
Missing text: 0
Missing labels: 0

TEST
label
0    592
1    557
Name: count, dtype: int64
Missing text: 0
Missing labels: 0


In [6]:
# Download resources required for word tokenization,
# part-of-speech tagging, and named entity recognition
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("maxent_ne_chunker_tab")
nltk.download("words")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker_tab.zip.
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.


True

In [7]:
import math

# Use all available named entities from each article
n = 100

def extract_ner_entities(text):
    # Tokenize the article into individual words and punctuation
    tokens = nltk.word_tokenize(text)

    # Assign a part-of-speech tag to each token
    tagged_words = nltk.pos_tag(tokens)

    # Identify named entities using NLTK's named entity chunker
    named_entities = nltk.ne_chunk(tagged_words)

    # Extract named entities, preserving multi-word entities
    ner_words = []

    for chunk in named_entities:
        if hasattr(chunk, "label"):
            ner_words.append(
                " ".join(c[0] for c in chunk.leaves())
            )

    # Calculate the maximum number of entities to retain
    total_words = len(tokens)
    num_ner_words = math.ceil(total_words * (n * 0.01))

    # Remove duplicate named entities, as in the original implementation
    ner_words = list(set(ner_words))

    # Limit the number of entities to those available
    if num_ner_words > len(ner_words):
        num_ner_words = len(ner_words)

    # Combine the extracted entities into a single sequence
    return " ".join(ner_words[:num_ner_words])

## NER preprocessing

In [8]:
from tqdm.auto import tqdm

# Enable progress bars for pandas operations
tqdm.pandas()

# Store the NER-preprocessed versions of both datasets
ner_preprocessed = {}

for dataset_name, splits in ner_raw.items():
    ner_preprocessed[dataset_name] = {}

    for split_name, df in splits.items():
        print(f"\nProcessing {dataset_name} - {split_name}")

        # Create a copy to preserve the original dataset
        processed_df = df.copy()

        # Replace full article text with extracted named entities
        processed_df["body"] = processed_df["body"].progress_apply(
            extract_ner_entities
        )

        # Store the processed dataset with its original labels
        ner_preprocessed[dataset_name][split_name] = processed_df

print("\nNER preprocessing complete!")


Processing ReCOVery - train


  0%|          | 0/1453 [00:00<?, ?it/s]


Processing ReCOVery - validation


  0%|          | 0/184 [00:00<?, ?it/s]


Processing ReCOVery - test


  0%|          | 0/392 [00:00<?, ?it/s]

... (3 outputs left out)

  0%|          | 0/1148 [00:00<?, ?it/s]


Processing Fake_And_Real_News - test


  0%|          | 0/1149 [00:00<?, ?it/s]


NER preprocessing complete!


In [9]:
import os

# Portable output directory
save_dir = "/content/SLIM_NER_results"
os.makedirs(save_dir, exist_ok=True)
print("Results directory:", save_dir)

Results directory: /content/SLIM_NER_results


In [10]:
# Calculate information reduction for each dataset split
stats = []

for dataset_name, splits in ner_preprocessed.items():
    for split_name, df in splits.items():

        # Count words in the original articles
        original_counts = ner_raw[dataset_name][split_name]["body"].apply(
            lambda text: len(text.split())
        )

        # Count words remaining after NER extraction
        ner_counts = df["body"].apply(
            lambda text: len(text.split())
        )

        # Calculate average word counts
        original_avg = original_counts.mean()
        ner_avg = ner_counts.mean()

        # Calculate percentage reduction in word count
        reduction = (1 - ner_avg / original_avg) * 100

        # Store statistics for each split
        stats.append({
            "Dataset": dataset_name,
            "Split": split_name,
            "Original Avg Words": original_avg,
            "NER Avg Words": ner_avg,
            "Reduction (%)": reduction,
            "Empty Samples": (ner_counts == 0).sum()
        })

# Display preprocessing statistics
ner_stats = pd.DataFrame(stats)
display(ner_stats.round(2))

,Dataset,Split,Original Avg Words,NER Avg Words,Reduction (%),Empty Samples
0,ReCOVery,train,825.60,42.83,94.81,4
1,ReCOVery,validation,921.71,51.98,94.36,0
2,ReCOVery,test,849.12,46.61,94.51,1
3,Fake_And_Real_News,train,817.41,48.48,94.07,56
4,Fake_And_Real_News,validation,768.54,47.01,93.88,27
5,Fake_And_Real_News,test,812.53,48.96,93.97,19


In [11]:
from transformers import XLNetTokenizer

# Specify the pretrained XLNet model and maximum sequence length
model_name = "xlnet-base-cased"
max_length = 512

# Initialize the tokenizer with the authors' lowercasing configuration
tokenizer = XLNetTokenizer.from_pretrained(
    model_name,
    do_lower_case=True
)

# Inspect tokenizer properties
print("Vocabulary size:", tokenizer.vocab_size)
print("Padding side:", tokenizer.padding_side)
print("Lowercasing:", tokenizer.do_lower_case)
print("Maximum sequence length:", max_length)

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

Vocabulary size: 32000
Padding side: left
Lowercasing: True
Maximum sequence length: 512


In [12]:
# Calculate average XLNet token counts before truncation and padding
token_stats = []

for dataset_name, splits in ner_preprocessed.items():
    for split_name, df in splits.items():

        # Count XLNet tokens in the original article text
        original_counts = ner_raw[dataset_name][split_name]["body"].apply(
            lambda text: len(tokenizer.tokenize(text))
        )

        # Count XLNet tokens in the extracted named entities
        ner_counts = df["body"].apply(
            lambda text: len(tokenizer.tokenize(text))
        )

        # Store token statistics for each dataset split
        token_stats.append({
            "Dataset": dataset_name,
            "Split": split_name,
            "Original Avg Tokens": original_counts.mean(),
            "NER Avg Tokens": ner_counts.mean(),
            "Reduction (%)": (
                1 - ner_counts.mean() / original_counts.mean()
            ) * 100,
            "NER Max Tokens": ner_counts.max(),
            "Samples Exceeding Limit": (ner_counts > max_length - 2).sum()
        })

# Display token-level statistics
ner_token_stats = pd.DataFrame(token_stats)
display(ner_token_stats.round(2))

,Dataset,Split,Original Avg Tokens,NER Avg Tokens,Reduction (%),NER Max Tokens,Samples Exceeding Limit
0,ReCOVery,train,1145.55,88.10,92.31,1219,11
1,ReCOVery,validation,1297.17,112.80,91.30,4780,2
2,ReCOVery,test,1180.81,97.32,91.76,3894,3
3,Fake_And_Real_News,train,1149.68,101.66,91.16,1648,25
4,Fake_And_Real_News,validation,1082.41,98.14,90.93,978,10
5,Fake_And_Real_News,test,1142.58,102.27,91.05,1219,13


In [13]:
from torch.utils.data import Dataset

class NERDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_length=512):

        # Match the original CSV-based pipeline, where empty
        # NER sequences are read as NaN and converted to "nan"
        texts = dataframe["body"].replace("", np.nan).astype(str).tolist()

        # Store binary classification labels
        self.labels = dataframe["label"].astype(int).tolist()

        # Tokenize, truncate, and pad each sequence to 512 tokens
        self.encodings = tokenizer(
            texts,
            add_special_tokens=True,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_attention_mask=True,
            return_token_type_ids=False,
            return_tensors="pt"
        )

    def __len__(self):
        # Return the total number of samples
        return len(self.labels)

    def __getitem__(self, idx):
        # Return the tensors required by XLNet for one sample
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [14]:
# Create tokenized PyTorch datasets for both news datasets
ner_datasets = {}

for dataset_name, splits in ner_preprocessed.items():
    ner_datasets[dataset_name] = {}

    for split_name, df in splits.items():

        # Convert extracted named entities into XLNet inputs
        ner_datasets[dataset_name][split_name] = NERDataset(
            dataframe=df,
            tokenizer=tokenizer,
            max_length=max_length
        )

        # Verify the number of samples
        print(
            dataset_name,
            split_name,
            len(ner_datasets[dataset_name][split_name])
        )

ReCOVery train 1453
ReCOVery validation 184
ReCOVery test 392
Fake_And_Real_News train 2297
Fake_And_Real_News validation 1148
Fake_And_Real_News test 1149


In [15]:
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

# Set the batch size used in the original implementation
batch_size = 16

# Store DataLoaders for both datasets
ner_dataloaders = {}

for dataset_name, splits in ner_datasets.items():
    ner_dataloaders[dataset_name] = {}

    for split_name, dataset in splits.items():

        # Randomize training samples and preserve evaluation order
        sampler = (
            RandomSampler(dataset)
            if split_name == "train"
            else SequentialSampler(dataset)
        )

        # Group individual samples into batches
        ner_dataloaders[dataset_name][split_name] = DataLoader(
            dataset,
            sampler=sampler,
            batch_size=batch_size
        )

        # Display the number of batches per split
        print(
            dataset_name,
            split_name,
            "Batches:", len(ner_dataloaders[dataset_name][split_name])
        )

ReCOVery train Batches: 91
ReCOVery validation Batches: 12
ReCOVery test Batches: 25
Fake_And_Real_News train Batches: 144
Fake_And_Real_News validation Batches: 72
Fake_And_Real_News test Batches: 72


In [16]:
import torch
import random
from transformers import XLNetForSequenceClassification

# Set random seeds for reproducibility
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Select the GPU for model training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained XLNet with a binary classification head
model = XLNetForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Move the model to the GPU
model = model.to(device)

# Inspect model properties
print("Device:", device)
print("Number of classes:", model.config.num_labels)
print("Hidden size:", model.config.d_model)
print("Number of layers:", model.config.n_layer)
print("Attention heads:", model.config.n_head)
print("Total parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  467MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

[transformers] XLNetForSequenceClassification LOAD REPORT from: xlnet-base-cased
Key                             | Status     | 
--------------------------------+------------+-
lm_loss.weight                  | UNEXPECTED | 
lm_loss.bias                    | UNEXPECTED | 
logits_proj.weight              | MISSING    | 
sequence_summary.summary.weight | MISSING    | 
logits_proj.bias                | MISSING    | 
sequence_summary.summary.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda
Number of classes: 2
Hidden size: 768
Number of layers: 12
Attention heads: 12
Total parameters: 117,310,466


In [17]:
from transformers import get_linear_schedule_with_warmup

# Define training hyperparameters
epochs = 15
learning_rate = 5e-5
epsilon = 1e-8
weight_decay = 0.0
warmup_steps = 0
max_grad_norm = 1.0

# Select ReCOVery as the first dataset for training
dataset_name = "ReCOVery"

train_dataloader = ner_dataloaders[dataset_name]["train"]
validation_dataloader = ner_dataloaders[dataset_name]["validation"]
test_dataloader = ner_dataloaders[dataset_name]["test"]

# Initialize the AdamW optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    eps=epsilon,
    weight_decay=weight_decay
)

# Calculate the total number of optimization steps
total_steps = len(train_dataloader) * epochs

# Initialize a scheduler that linearly decreases the learning rate
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("Dataset:", dataset_name)
print("Epochs:", epochs)
print("Batch size:", batch_size)
print("Learning rate:", learning_rate)
print("Total training steps:", total_steps)

Dataset: ReCOVery
Epochs: 15
Batch size: 16
Learning rate: 5e-05
Total training steps: 1365


In [18]:
import time

def train_one_epoch(model, dataloader, optimizer, scheduler):

    # Enable training mode, including dropout
    model.train()

    total_loss = 0
    start_time = time.time()

    # Iterate through all training batches
    for batch in tqdm(dataloader, desc="Training"):

        # Move the current batch to the GPU
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        # Clear gradients from the previous training step
        optimizer.zero_grad(set_to_none=True)

        # Forward propagation and cross-entropy loss calculation
        outputs = model(**batch)
        loss = outputs.loss

        # Accumulate batch loss
        total_loss += loss.item()

        # Backpropagate to calculate parameter gradients
        loss.backward()

        # Clip gradients to prevent excessively large updates
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_grad_norm
        )

        # Update model parameters
        optimizer.step()

        # Update the learning rate
        scheduler.step()

    # Calculate average training loss across batches
    avg_train_loss = total_loss / len(dataloader)

    # Calculate total training time for the epoch
    elapsed_time = time.time() - start_time

    return avg_train_loss, elapsed_time

In [19]:
def validate(model, dataloader):

    # Enable evaluation mode and disable dropout
    model.eval()

    total_accuracy = 0
    start_time = time.time()

    # Disable gradient calculation during validation
    with torch.no_grad():

        for batch in tqdm(dataloader, desc="Validation"):

            # Move the current batch to the GPU
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            # Forward propagation without calculating loss
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            # Convert logits into predicted class labels
            predictions = torch.argmax(outputs.logits, dim=1)

            # Calculate accuracy for the current batch
            batch_accuracy = (
                (predictions == labels).float().mean().item()
            )

            # Accumulate batch accuracy
            total_accuracy += batch_accuracy

    # Average the batch accuracies, following the original code
    avg_accuracy = total_accuracy / len(dataloader)

    # Calculate validation time
    elapsed_time = time.time() - start_time

    return avg_accuracy, elapsed_time

## Experiment 1: ReCOVery (NER + XLNet)

In [20]:
import json
import os

# Create a directory for the ReCOVery experiment
experiment_dir = os.path.join(save_dir, "ReCOVery")
os.makedirs(experiment_dir, exist_ok=True)

# Define paths for the best checkpoint and training history
best_model_path = os.path.join(experiment_dir, "best_model.pth")
history_path = os.path.join(experiment_dir, "training_history.csv")

# Reset random seeds before training
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Initialize training history and checkpoint tracking
training_history = []
best_accuracy = -0.1
best_epoch = None

# Train the model for 15 epochs
for epoch in range(epochs):

    print(f"\n========== Epoch {epoch + 1}/{epochs} ==========")

    # Train the model for one epoch
    train_loss, train_time = train_one_epoch(
        model,
        train_dataloader,
        optimizer,
        scheduler
    )

    # Evaluate the model on the validation dataset
    val_accuracy, val_time = validate(
        model,
        validation_dataloader
    )

    # Display epoch results
    print(f"Training loss: {train_loss:.4f}")
    print(f"Validation accuracy: {val_accuracy:.4f}")
    print(f"Training time: {train_time / 60:.2f} minutes")
    print(f"Validation time: {val_time / 60:.2f} minutes")

    # Save the model whenever validation accuracy improves
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_epoch = epoch + 1

        torch.save(model.state_dict(), best_model_path)

        print("New best model saved!")

    # Record epoch-level training statistics
    training_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "validation_accuracy": val_accuracy,
        "learning_rate": scheduler.get_last_lr()[0],
        "training_time_seconds": train_time,
        "validation_time_seconds": val_time,
        "best_epoch": best_epoch
    })

    # Back up training history after every epoch
    pd.DataFrame(training_history).to_csv(
        history_path, index=False
    )

print("\nTraining complete!")
print("Best epoch:", best_epoch)
print(f"Best validation accuracy: {best_accuracy:.4f}")


========== Epoch 1/15 ==========


Training:   0%|          | 0/91 [00:00<?, ?it/s]

Validation:   0%|          | 0/12 [00:00<?, ?it/s]

Training loss: 0.6556
Validation accuracy: 0.6354
Training time: 0.30 minutes
Validation time: 0.01 minutes
New best model saved!

========== Epoch 2/15 ==========


Training:   0%|          | 0/91 [00:00<?, ?it/s]

Validation:   0%|          | 0/12 [00:00<?, ?it/s]

... (36 outputs left out)

Training loss: 0.0762
Validation accuracy: 0.8698
Training time: 0.29 minutes
Validation time: 0.01 minutes

========== Epoch 15/15 ==========


Training:   0%|          | 0/91 [00:00<?, ?it/s]

Validation:   0%|          | 0/12 [00:00<?, ?it/s]

Training loss: 0.0696
Validation accuracy: 0.8698
Training time: 0.29 minutes
Validation time: 0.01 minutes

Training complete!
Best epoch: 7
Best validation accuracy: 0.8698


### ReCOVery — executable checkpoint evaluation

When running from scratch, this evaluates the checkpoint saved by the ReCOVery training cell.

In [21]:
#getting metrics for recovery
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Reload the checkpoint with the highest validation accuracy
model.load_state_dict(
    torch.load(best_model_path, map_location=device)
)

# Enable evaluation mode
model.eval()

# Store predictions, probabilities, and ground-truth labels
all_predictions = []
all_probabilities = []
all_labels = []

# Generate predictions without updating model parameters
with torch.no_grad():

    for batch in tqdm(test_dataloader, desc="Testing"):

        # Move model inputs to the GPU
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Generate classification logits
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Convert logits into predicted classes and probabilities
        predictions = torch.argmax(outputs.logits, dim=1)
        probabilities = torch.softmax(outputs.logits, dim=1)[:, 1]

        # Store results on the CPU
        all_predictions.extend(predictions.cpu().tolist())
        all_probabilities.extend(probabilities.cpu().tolist())
        all_labels.extend(batch["labels"].tolist())

# Calculate test performance metrics
test_results = {
    "accuracy": accuracy_score(all_labels, all_predictions),
    "macro_f1": f1_score(
        all_labels, all_predictions, average="macro"
    ),
    "auc_paper": roc_auc_score(all_labels, all_predictions),
    "auc_probability": roc_auc_score(all_labels, all_probabilities)
}

# Display test results
print("\nTest Results")

for metric, value in test_results.items():
    print(f"{metric}: {value:.4f}")

# Display class-level performance
print("\nClassification Report")
print(classification_report(
    all_labels,
    all_predictions,
    target_names=["Fake", "Real"],
    digits=4
))

# Display the confusion matrix
print("\nConfusion Matrix")
print(confusion_matrix(all_labels, all_predictions))

Testing:   0%|          | 0/25 [00:00<?, ?it/s]


Test Results
accuracy: 0.8801
macro_f1: 0.8565
auc_paper: 0.8611
auc_probability: 0.9231

Classification Report
              precision    recall  f1-score   support

        Fake     0.7815    0.8158    0.7983       114
        Real     0.9231    0.9065    0.9147       278

    accuracy                         0.8801       392
   macro avg     0.8523    0.8611    0.8565       392
weighted avg     0.8819    0.8801    0.8808       392


Confusion Matrix
[[ 93  21]
 [ 26 252]]


In [22]:
# Save final test metrics and experiment details
experiment_results = {
    "dataset": dataset_name,
    "model": model_name,
    "method": "NER",
    "seed": seed,
    "epochs": epochs,
    "batch_size": batch_size,
    "max_length": max_length,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "best_epoch": best_epoch,
    "best_validation_accuracy": best_accuracy,
    "test_results": test_results
}

with open(os.path.join(experiment_dir, "test_results.json"), "w") as f:
    json.dump(experiment_results, f, indent=4)

# Save individual test predictions
pd.DataFrame({
    "true_label": all_labels,
    "predicted_label": all_predictions,
    "probability_real": all_probabilities
}).to_csv(
    os.path.join(experiment_dir, "test_predictions.csv"),
    index=False
)

# Save the tokenizer and model configuration
tokenizer.save_pretrained(experiment_dir)
model.config.save_pretrained(experiment_dir)

# Verify the saved experiment files
for filename in os.listdir(experiment_dir):
    print(filename)

best_model_smoke.pth
test_results.json
test_predictions.csv
tokenizer.json
training_history_smoke.csv
tokenizer_config.json
config.json


## Experiment 2: Fake and Real News (NER + XLNet)

The original successful run used a legacy Hugging Face AdamW implementation. The setup below preserves that implementation and the completed 15-epoch output.

In [23]:
import requests
import ast
import math
import warnings
import torch

from typing import Callable, Iterable, Tuple
from torch import nn
from torch.optim import Optimizer
from transformers.utils.versions import require_version

# Retrieve the legacy Hugging Face optimizer source
url = (
    "https://raw.githubusercontent.com/huggingface/"
    "transformers/v4.30.2/src/transformers/optimization.py"
)

response = requests.get(url, timeout=30)
response.raise_for_status()

# Extract only the AdamW class without changing Transformers
source_tree = ast.parse(response.text)

adamw_class = next(
    node for node in source_tree.body
    if isinstance(node, ast.ClassDef) and node.name == "AdamW"
)

# Load the original optimizer implementation
legacy_module = ast.Module(
    body=[adamw_class],
    type_ignores=[]
)

exec(compile(legacy_module, "<legacy_adamw>", "exec"))

LegacyAdamW = AdamW

print("Legacy Hugging Face AdamW loaded successfully!")

Legacy Hugging Face AdamW loaded successfully!


In [24]:
import gc
import random
import torch
import numpy as np

from transformers import (
    XLNetForSequenceClassification,
    get_linear_schedule_with_warmup
)

# Release the previous model and optimizer from GPU memory
for variable in ["model", "optimizer", "scheduler", "outputs", "loss"]:
    globals().pop(variable, None)

gc.collect()
torch.cuda.empty_cache()

# Preserve the previous experiment's hyperparameters
seed = 42
epochs = 15
learning_rate = 5e-5
epsilon = 1e-8
weight_decay = 0.0
warmup_steps = 0
max_grad_norm = 1.0

# Set random seeds for reproducibility
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Initialize a fresh XLNet classification model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = XLNetForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
).to(device)

# Select the Fake and Real News dataset
dataset_name = "Fake_And_Real_News"

train_dataloader = ner_dataloaders[dataset_name]["train"]
validation_dataloader = ner_dataloaders[dataset_name]["validation"]
test_dataloader = ner_dataloaders[dataset_name]["test"]

# Use the legacy Hugging Face AdamW optimizer
optimizer = LegacyAdamW(
    model.parameters(),
    lr=learning_rate,
    eps=epsilon,
    weight_decay=weight_decay,
    correct_bias=True
)

# Preserve the linear learning rate decay schedule
total_steps = len(train_dataloader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# Verify the experimental configuration
print("Device:", device)
print("Optimizer:", type(optimizer).__name__)
print("Legacy optimizer:", isinstance(optimizer, LegacyAdamW))
print("Learning rate:", scheduler.get_last_lr())
print("Total training steps:", total_steps)
print("Seed:", seed)

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

[transformers] XLNetForSequenceClassification LOAD REPORT from: xlnet-base-cased
Key                             | Status     | 
--------------------------------+------------+-
lm_loss.weight                  | UNEXPECTED | 
lm_loss.bias                    | UNEXPECTED | 
logits_proj.weight              | MISSING    | 
sequence_summary.summary.weight | MISSING    | 
logits_proj.bias                | MISSING    | 
sequence_summary.summary.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda
Optimizer: AdamW
Legacy optimizer: True
Learning rate: [5e-05]
Total training steps: 2160
Seed: 42


<legacy_adamw>:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning


In [25]:
from tqdm.auto import tqdm
import os
import random
import pandas as pd

# Create a separate local directory for this experiment
experiment_dir = os.path.join(
    save_dir, "Fake_And_Real_News_legacy_AdamW"
)

os.makedirs(experiment_dir, exist_ok=True)

best_model_path = os.path.join(experiment_dir, "best_model.pth")
history_path = os.path.join(experiment_dir, "training_history.csv")

# Reset random seeds before training
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Initialize variables for tracking training progress
training_history = []
best_accuracy = -0.1
best_epoch = None

# Train and validate the model across all epochs
for epoch in range(epochs):

    print(f"\n========== Epoch {epoch + 1}/{epochs} ==========")

    train_loss, train_time = train_one_epoch(
        model, train_dataloader, optimizer, scheduler
    )

    val_accuracy, val_time = validate(
        model, validation_dataloader
    )

    print(f"Training loss: {train_loss:.4f}")
    print(f"Validation accuracy: {val_accuracy:.4f}")
    print(f"Training time: {train_time / 60:.2f} minutes")
    print(f"Validation time: {val_time / 60:.2f} minutes")

    # Save the model whenever validation accuracy improves
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_epoch = epoch + 1

        torch.save(model.state_dict(), best_model_path)
        print("New best model saved!")

    # Record the results of each completed epoch
    training_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "validation_accuracy": val_accuracy,
        "learning_rate": scheduler.get_last_lr()[0],
        "training_time_seconds": train_time,
        "validation_time_seconds": val_time,
        "best_epoch": best_epoch
    })

    # Save progress after every epoch
    pd.DataFrame(training_history).to_csv(
        history_path, index=False
    )

print("\nTraining complete!")
print("Best epoch:", best_epoch)
print(f"Best validation accuracy: {best_accuracy:.4f}")


========== Epoch 1/15 ==========


Training:   0%|          | 0/144 [00:00<?, ?it/s]

Validation:   0%|          | 0/72 [00:00<?, ?it/s]

Training loss: 0.5369
Validation accuracy: 0.7911
Training time: 0.46 minutes
Validation time: 0.08 minutes
New best model saved!

========== Epoch 2/15 ==========


Training:   0%|          | 0/144 [00:00<?, ?it/s]

Validation:   0%|          | 0/72 [00:00<?, ?it/s]

... (36 outputs left out)

Training loss: 0.1548
Validation accuracy: 0.8808
Training time: 0.46 minutes
Validation time: 0.08 minutes

========== Epoch 15/15 ==========


Training:   0%|          | 0/144 [00:00<?, ?it/s]

Validation:   0%|          | 0/72 [00:00<?, ?it/s]

Training loss: 0.1453
Validation accuracy: 0.8834
Training time: 0.46 minutes
Validation time: 0.08 minutes
New best model saved!

Training complete!
Best epoch: 15
Best validation accuracy: 0.8834


In [26]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm

# Restore the model with the highest validation accuracy
model.load_state_dict(
    torch.load(best_model_path, map_location=device)
)

model.eval()

# Store predictions, probabilities and ground-truth labels
all_predictions = []
all_probabilities = []
all_labels = []

# Evaluate the best model on the unseen test set
with torch.no_grad():
    for batch in tqdm(test_dataloader, desc="Testing"):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Obtain predicted classes and real-news probabilities
        predictions = torch.argmax(outputs.logits, dim=1)
        probabilities = torch.softmax(outputs.logits, dim=1)[:, 1]

        all_predictions.extend(predictions.cpu().tolist())
        all_probabilities.extend(probabilities.cpu().tolist())
        all_labels.extend(batch["labels"].tolist())

# Calculate evaluation metrics
test_results = {
    "accuracy": accuracy_score(all_labels, all_predictions),
    "macro_f1": f1_score(
        all_labels, all_predictions, average="macro"
    ),
    "auc_paper": roc_auc_score(all_labels, all_predictions),
    "auc_probability": roc_auc_score(all_labels, all_probabilities)
}

for metric, value in test_results.items():
    print(f"{metric}: {value:.4f}")

# Display class-specific performance
print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_predictions,
    target_names=["Fake", "Real"],
    digits=4
))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_predictions))

Testing:   0%|          | 0/72 [00:00<?, ?it/s]

accuracy: 0.8903
macro_f1: 0.8903
auc_paper: 0.8904
auc_probability: 0.9541

Classification Report:
              precision    recall  f1-score   support

        Fake     0.8990    0.8868    0.8929       592
        Real     0.8814    0.8941    0.8877       557

    accuracy                         0.8903      1149
   macro avg     0.8902    0.8904    0.8903      1149
weighted avg     0.8905    0.8903    0.8904      1149


Confusion Matrix:
[[525  67]
 [ 59 498]]


In [27]:
import os
import json
import pandas as pd

# Record the experimental configuration and test results
experiment_results = {
    "dataset": dataset_name,
    "model": "xlnet-base-cased",
    "method": "NER",
    "optimizer": "Legacy Hugging Face AdamW",
    "seed": seed,
    "epochs": epochs,
    "batch_size": batch_size,
    "max_length": max_length,
    "learning_rate": learning_rate,
    "epsilon": epsilon,
    "weight_decay": weight_decay,
    "best_epoch": best_epoch,
    "best_validation_accuracy": best_accuracy,
    "test_results": test_results
}

# Save the configuration and evaluation metrics
with open(os.path.join(experiment_dir, "test_results.json"), "w") as f:
    json.dump(experiment_results, f, indent=4)

# Save individual test predictions
pd.DataFrame({
    "true_label": all_labels,
    "predicted_label": all_predictions,
    "probability_real": all_probabilities
}).to_csv(
    os.path.join(experiment_dir, "test_predictions.csv"),
    index=False
)

# Save the tokenizer and model configuration
tokenizer.save_pretrained(experiment_dir)
model.config.save_pretrained(experiment_dir)

# Verify all experiment files
print("Saved experiment files:")
for filename in os.listdir(experiment_dir):
    print(filename)

Saved experiment files:
best_model.pth
training_history.csv
test_results.json
test_predictions.csv
tokenizer.json
tokenizer_config.json
config.json
